In [1]:
import pandas as pd
import lightgbm as lgb
import time
import joblib
import gc
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
from scipy.stats import entropy

filenames = {
    "Original":         "cleaned_original_data.csv",
    "Small GAN":        "small_GAN_combined.csv",
    "Med GAN":          "med_GAN_combined.csv",
    "Large GAN":        "large_GAN_combined.csv",
    "Small Diffusion":  "small_Diffusion_combined.csv",
    "Med Diffusion":    "med_Diffusion_combined.csv",
    "Large Diffusion":  "large_Diffusion_combined.csv",
    "Small LLM":        "small_LLM_combined.csv",
    "Med LLM":          "med_LLM_combined.csv",
    # "Large LLM":        "large_LLM_combined.csv",
}

gc.collect()
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
target = 'class1'
drop_cols = ['class2', 'class3']

df_orig = pd.read_csv(filenames["Original"])
X_all_orig = df_orig.drop(columns=[target] + drop_cols, errors='ignore')
y_all_orig = df_orig[target]

X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_all_orig, y_all_orig, test_size=0.2, random_state=42
)
all_possible_labels = sorted(y_all_orig.unique())

orig_dist = (pd.Series(y_train_orig).value_counts(normalize=True)
             .reindex(all_possible_labels, fill_value=0)
             .sort_index().values)

num_classes = y_all_orig.nunique()
encoders = joblib.load('label_encoders.joblib')
le = encoders['class1']
results = []
dist_data = []

for name, path in filenames.items():
    print(f"Processing {name}...")
    start_time = time.time()

    if name == "Original":
        X_train, y_train = X_train_orig, y_train_orig
    else:
        df_temp = pd.read_csv(path)
        y_train_raw = pd.Series(df_temp[target]).astype(pd.CategoricalDtype(categories=all_possible_labels))
        X_train_raw = df_temp.reindex(columns=X_test_orig.columns)
        mask = y_train_raw.notna()
        if not mask.all():
            invalid_count = (~mask).sum()
            print(f"⚠️ Warning: {name} contained {invalid_count} unrecognized labels. Dropping those rows.")
            y_train = y_train_raw[mask]
            X_train = X_train_raw[mask]
        else:
            y_train = y_train_raw
            X_train = X_train_raw
        X_train = X_train.fillna(0)
        del df_temp

    current_dist = (y_train.value_counts(normalize=True)
                    .reindex(all_possible_labels, fill_value=0)
                    .sort_index().values)
    kl_div = entropy(orig_dist, current_dist + 1e-10)
    counts = y_train.value_counts(normalize=True)
    for label_idx, ratio in counts.items():
        # Map the integer back to the string name
        class_name = le.inverse_transform([int(label_idx)])[0]

        dist_data.append({
            "Dataset": name,
            "Class": class_name,
            "Percentage": ratio * 100
        })

    model = lgb.LGBMClassifier(
        min_data_in_leaf=1,
        min_sum_hessian_in_leaf=0.001,
        zero_as_missing=True,
        objective='multiclass',
        num_class=num_classes,
        device='cpu',
        gpu_platform_id=0,
        gpu_device_id=0,
        n_estimators=200,
        learning_rate=0.05,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test_orig)
    y_prob_raw = model.predict_proba(X_test_orig)
    y_prob_full = np.zeros((len(X_test_orig), num_classes))
    classes_seen = model.classes_
    
    for i, class_val in enumerate(classes_seen):
        col_idx = all_possible_labels.index(class_val)
        y_prob_full[:, col_idx] = y_prob_raw[:, i]
        
    results.append({
        "Dataset": name,
        "KL Divergence": round(kl_div, 4),
        "Accuracy": round(accuracy_score(y_test_orig, y_pred), 4),
        "F1 (Macro)": round(f1_score(y_test_orig, y_pred, average='macro'), 4),
        "AUC (OvR)": round(roc_auc_score(y_test_orig, y_prob_full, multi_class='ovr'), 4)
    })

    if name == "Original":
        del X_train, y_train
        del X_train_orig, y_train_orig
    else:
        del X_train, y_train

    gc.collect()
df_res = pd.DataFrame(results)

Processing Original...
Processing Small GAN...
Processing Med GAN...
Processing Large GAN...
Processing Small Diffusion...
Processing Med Diffusion...
Processing Large Diffusion...
Processing Small LLM...
Processing Med LLM...


In [2]:
print(df_res)
print(dist_data)


def split_name(name):
    if name == "Original":
        return "Original", "Original"
    parts = name.split(" ")
    return parts[0], parts[1]

df_res[['Size', 'Method']] = df_res['Dataset'].apply(lambda x: pd.Series(split_name(x)))

df_res = df_res[['Method', 'Size', 'Accuracy', 'F1 (Macro)', 'AUC (OvR)', 'KL Divergence']]

print(df_res)
df_res.to_csv('pivoted_results_full.csv')

df_dist = pd.DataFrame(dist_data)
df_dist['Proportion'] = df_dist['Percentage'] / 100


orig_df = df_dist[df_dist['Dataset'] == 'Original'].set_index('Class')
original_dist = orig_df['Proportion']
dataset_names = list(filenames.keys())


threshold = 0.01  # 1%
common_classes = original_dist[original_dist > threshold].index.tolist()
rare_classes = original_dist[original_dist <= threshold].index.tolist()


df_runtimes = pd.read_csv('runtimes_log.csv')
df_runtimes[['Size', 'Method']] = df_runtimes['Model'].str.split(' ', expand=True)

size_order = ['Small', 'Med', 'Large']
method_order = ['GAN', 'Diffusion', 'LLM']


df_runtimes['Size'] = pd.Categorical(df_runtimes['Size'], categories=size_order, ordered=True)
df_runtimes['Method'] = pd.Categorical(df_runtimes['Method'], categories=method_order, ordered=True)
df_runtimes = df_runtimes.sort_values(['Method', 'Size'])


           Dataset  KL Divergence  Accuracy  F1 (Macro)  AUC (OvR)
0         Original         0.0000    0.5401      0.1402     0.5412
1        Small GAN         0.2214    0.9495      0.5277     0.8228
2          Med GAN         0.0008    0.4997      0.0878     0.5197
3        Large GAN         0.0006    0.5635      0.2678     0.6495
4  Small Diffusion         1.7351    0.1041      0.0206     0.5501
5    Med Diffusion         0.9397    0.2623      0.0477     0.5370
6  Large Diffusion         0.9577    0.1352      0.0487     0.4843
7        Small LLM         0.2214    0.9496      0.5277     0.8228
8          Med LLM         0.0008    0.3880      0.1914     0.6214
[{'Dataset': 'Original', 'Class': 'Normal', 'Percentage': 51.33713130094858}, {'Dataset': 'Original', 'Class': 'RDOS', 'Percentage': 17.224102931927447}, {'Dataset': 'Original', 'Class': 'Scanning_vulnerability', 'Percentage': 6.434311454664236}, {'Dataset': 'Original', 'Class': 'Generic_scanning', 'Percentage': 6.11512379942954

In [3]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

orig_metrics = df_res[df_res['Method'] == 'Original'].iloc[0]
df_plot = df_res[df_res['Method'] != 'Original'].copy()

df_melted = df_plot.melt(id_vars=['Method', 'Size'],
                         value_vars=['Accuracy', 'F1 (Macro)', 'AUC (OvR)', 'KL Divergence'],
                         var_name='Metric', value_name='Score')

fig = px.bar(
    df_melted,
    x="Method",
    y="Score",
    color="Size",
    barmode="group",
    facet_col="Metric",
    category_orders={"Size": ["Small", "Med", "Large"],
                     "Method": ["GAN", "Diffusion", "LLM"]},
    title="Synthetic Data Utility: Comparison of Generation Methods and Sample Sizes",
    labels={"Score": "Performance Score (on Real Test Set)"},
    height=500
)

for i, metric in enumerate(['Accuracy', 'F1 (Macro)', 'AUC (OvR)', 'KL Divergence']):
    
    fig.add_hline(y=orig_metrics[metric], line_dash="dash", line_color="red",
                  annotation_text=f"Orig {metric}",annotation_position="top left",
        col=i+1)

fig.write_image("synthetic_comparison.svg", width=1200, height=600)
fig.show()

all_classes = sorted(df_dist['Class'].unique())

fig_dist = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Common Classes (>1%) - Log Scale", "Rare Classes (<1%) - Log Scale"),
    vertical_spacing=0.12,
    row_heights=[0.4, 0.6]
)


def get_vals(cls_name):
    return [
        df_dist[(df_dist['Dataset'] == ds) & (df_dist['Class'] == cls_name)]['Proportion'].values[0] 
        if not df_dist[(df_dist['Dataset'] == ds) & (df_dist['Class'] == cls_name)].empty else 0 
        for ds in dataset_names
    ]


for cls in common_classes:
    fig_dist.add_trace(go.Bar(
        x=dataset_names, y=get_vals(cls), name=str(cls),
        legendgroup="common", legendgrouptitle_text="Common"
    ), row=1, col=1)


for cls in rare_classes:
    fig_dist.add_trace(go.Bar(
        x=dataset_names, y=get_vals(cls), name=str(cls),
        legendgroup="rare", legendgrouptitle_text="Rare Attacks"
    ), row=2, col=1)


fig_dist.update_yaxes(type="log", row=1, col=1, tickformat=".1%")
fig_dist.update_yaxes(type="log", row=2, col=1, tickformat=".2%")


fig_dist.update_layout(
    height=900,
    width=1100,
    barmode='group',
    template="plotly_white",
    title_text="Fidelity Analysis: Multi-Scale Logarithmic View",
    showlegend=True
)

fig_dist.write_image("class_distribution_fidelity.svg")
fig_dist.show()


def format_seconds(seconds):
    if pd.isna(seconds) or seconds == 0:
        return "-"
    h, m, s = int(seconds // 3600), int((seconds % 3600) // 60), int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


actual_counts = {}
for name, path in filenames.items():
    try:
        actual_counts[name] = len(pd.read_csv(path, usecols=[0]))
    except Exception:
        actual_counts[name] = 0


perf_df = df_runtimes.groupby('Model')['Runtime'].mean().reset_index()
perf_df['Rows'] = perf_df['Model'].map(actual_counts)

perf_df[['Size', 'Method']] = perf_df['Model'].str.split(' ', expand=True)


def combine_metrics(row):
    time_str = format_seconds(row['Runtime'])
    rows_str = f"{int(row['Rows']):,}" if not pd.isna(row['Rows']) else "0"
    return f"{time_str} | {rows_str}"

perf_df['Cell_Value'] = perf_df.apply(combine_metrics, axis=1)

final_table = perf_df.pivot(index='Method', columns='Size', values='Cell_Value')


size_order = ['Small', 'Med', 'Large']
size_mapping = {
    'Small': 'Small (800)',
    'Med': 'Med (8k)',
    'Large': 'Large (80k)'
}


existing_cols = [c for c in size_order if c in final_table.columns]
final_table = final_table[existing_cols]


final_table.columns = [size_mapping[c] for c in final_table.columns]


method_order = ['GAN', 'Diffusion', 'LLM']
existing_rows = [r for r in method_order if r in final_table.index]
final_table = final_table.reindex(existing_rows)

print("\n--- Model Generation Summary (Time | Total Rows) ---")
print(final_table)


final_table.to_csv("performance_summary_grid.csv")



--- Model Generation Summary (Time | Total Rows) ---
              Small (800)          Med (8k)        Large (80k)
Method                                                        
GAN        00:00:15 | 624  00:04:05 | 7,711  00:09:17 | 78,065
Diffusion  00:00:27 | 800  00:00:34 | 8,000  00:00:46 | 80,000
LLM        04:03:54 | 624  14:41:21 | 7,711                NaN
